# 5. 结构化输出

- 许多大模型，除了文本生成外，一般还支持：
    - **工具调用**
        - 调用外部工具（如数据库查询或 API 调用），并将结果用于生成响应。
    - **结构化输出**
        - 模型的响应被限制为遵循预定义的格式。
    - **多模态**
        - 处理并返回除文本以外的数据，例如图像、音频和视频。
    - **推理**
        - 模型执行多步推理以得出结论。

- 本部分内容主要介绍langchain中对结构化输出的处理。

## 5.1. 模型的结构化输出

- 可以要求模型提供与给定模式匹配格式的响应。这对于确保输出能够被轻松解析并用于后续处理非常有用。LangChain支持多种模式类型和方法来强制执行结构化输出。

### (1) Pydantic结构化定义

- 结构化模式：使用pydantic框架定义结构化(`from pydantic import BaseModel, Field`)
    - 继承BaseModel。
        - class  XXX(BaseModel)
    - 使用Field定义输出字段。
        - 数据字段：类型 = Field(description="描述...")
    

- BaseModel说明：
    - BaseModel主要提供基础的json与字典格式的转换。
        - model_dump：dict：转换为字典，不推荐使用dict()
        - model_dump_json：转换为json对象。不推荐json()
        - model_json_schema：模型的结构模式，不推荐schema与schema_json
    - 以及其他来源的数据分析。
        - parse_file：
        - parse_obj：
        - parse_raw：

- Field的核心是__call__可调用运算符，其中重要的是如下几个属性：
    - default：指定缺省值
    - description：设置字段的描述，这个描述会被大模型检测，并推导。
    - title：可阅读的名字。

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """电影的详细描述"""
    title: str = Field("Gone With Wind", description="电影名")
    year: int = Field(default=2, description="电影发布的年份")
    director: str = Field("张导", description="电影拍摄的导演", title="导演")
    rating: float = Field(8.5, description="电影的评分")

print(Movie().model_dump())     # dict类型。字段需要有值
print(Movie().model_dump_json())  # 字符串类型。
print(Movie().model_json_schema())  # 字符串类型。

{'title': 'Gone With Wind', 'year': 2, 'director': '张导', 'rating': 8.5}
{"title":"Gone With Wind","year":2,"director":"张导","rating":8.5}
{'description': '电影的详细描述', 'properties': {'title': {'default': 'Gone With Wind', 'description': '电影名', 'title': 'Title', 'type': 'string'}, 'year': {'default': 2, 'description': '电影发布的年份', 'title': 'Year', 'type': 'integer'}, 'director': {'default': '张导', 'description': '电影拍摄的导演', 'title': '导演', 'type': 'string'}, 'rating': {'default': 8.5, 'description': '电影的评分', 'title': 'Rating', 'type': 'number'}}, 'title': 'Movie', 'type': 'object'}


- 在模型中使用例子：

In [3]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class Movie(BaseModel):
    """电影的详细描述"""
    title: str = Field("Gone With Wind", description="电影名")
    year: int = Field(default=2, description="电影发布的年份")
    director: str = Field("张导", description="电影拍摄的导演", title="导演")
    rating: float = Field(8.5, description="电影的评分")

model = init_chat_model("ollama:qwen3.5:4b")
model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("关于中国大陆地区电影《1942》的信息")
print(response) 

KeyboardInterrupt: 

- 说明：
    - langchain的模型通过with_structured_output来提供结构化输出。
    - `Optional[str]`用来设置可选字段。

### (2) TypeDict结构化定义

In [ ]:
from typing_extensions import TypedDict, Annotated
from langchain.chat_models import init_chat_model
class MovieDict(TypedDict):
    """电影的详细描述"""
    title: Annotated[str, ..., "电影名",]
    year: Annotated[int, 2012, "电影发布的年份"]
    director: Annotated[str, ..., "电影拍摄的导演"]
    rating: Annotated[float, ..., "电影的评分"]

model = init_chat_model("ollama:qwen3.5:4b")
model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("帮我按照格式要求提供中国大陆地区电影《一九四二》的信息")
print(response) 

{'title': '一九四二', 'director': '张艺谋', 'rating': 7.8, 'year': 2012}


- 代码说明：
    - 在 Pydantic V2 中，Annotated 的第一个元数据参数可以用于传递 Field 对象。当使用 ... 时，它等价于 Field(required=True)，表示该字段是必需的。
    - 而 None 或其他值表示有默认值。

### (3) JSON Schema结构化定义

In [6]:
import json
from langchain.chat_models import init_chat_model
json_schema = {
    "title": "Movie",
    "description": "电影的详细描述",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "电影名"
        },
        "year": {
            "type": "integer",
            "description": "电影发行的年月日"
        },
        "director": {
            "type": "string",
            "description": "导演"
        },
        "rating": {
            "type": "number",
            "description": "电影的豆瓣评分"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

# model = init_chat_model("ollama:qwen3.5:9b")
model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke([{"role":"user", "content":"帮我按照格式要求提供中国大陆地区电影《一九四二》的信息"}])
print(response) 

{'title': '一九四二', 'year': 2012, 'director': '陈可辛', 'rating': 8.500000000000002}


- 代码说明：
    - model本身也是继承BaseModel的。
    - with_structured_output函数需要指定具体的结构化方法：`method="json_schema"`
        - 某些提供商支持不同的结构化输出方法:
            - 'json_schema'：使用提供商提供的专用结构化输出功能。
            - 'function_calling'：通过强制进行遵循给定模式的工具调用来获得结构化输出。
            - 'json_mode'：某些提供商提供的 'json_schema' 的前身。能够生成有效的 JSON，但必须在提示中描述模式。

### (4) 结构化输出的输出设置

- method参数设置对输出的格式控制

In [1]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class Movie(BaseModel):
    """电影的详细描述"""
    title: str = Field("Gone With Wind", description="电影名")
    year: int = Field(default=2, description="电影发布的年份")
    director: str = Field("张导", description="电影拍摄的导演", title="导演")
    rating: float = Field(8.5, description="电影的评分")

model = init_chat_model("ollama:qwen3:4b")
model_with_structure = model.with_structured_output(Movie, method="function_calling")
response = model_with_structure.invoke("关于中国大陆地区电影《1942》的信息")
print(response) 

title='1942' year=2008 director='陈可辛' rating=8.2


In [3]:
print(type(response))
response.title

<class '__main__.Movie'>


'1942'

- 包含原始输出控制
    - 设置include_raw=True以同时获取解析后的输出和原始 AI 消息。

In [4]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class Movie(BaseModel):
    """电影的详细描述"""
    title: str = Field("Gone With Wind", description="电影名")
    year: int = Field(default=2, description="电影发布的年份")
    director: str = Field("张导", description="电影拍摄的导演", title="导演")
    rating: float = Field(8.5, description="电影的评分")

model = init_chat_model("ollama:qwen3:4b")
model_with_structure = model.with_structured_output(Movie, method="function_calling", include_raw=True)
response = model_with_structure.invoke("关于中国大陆地区电影《1942》的信息")
print(response) 
print(type(response)) 

{'raw': AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-04-11T14:07:42.3507035Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11483451500, 'load_duration': 2648827200, 'prompt_eval_count': 180, 'prompt_eval_duration': 55135700, 'eval_count': 637, 'eval_duration': 8641171900, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019d7cde-8431-7d12-963e-0747a4c35517-0', tool_calls=[{'name': 'Movie', 'args': {'title': '1942'}, 'id': '7a0d98fd-a5a6-49d7-bf14-6d652b0ea37f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 180, 'output_tokens': 637, 'total_tokens': 817}), 'parsed': Movie(title='1942', year=2, director='张导', rating=8.5), 'parsing_error': None}
<class 'dict'>


- 代码说明：
    - 在获取解析后的结构化输出的同时，一并返回原始的 AIMessage 对象会非常有用，这样可以访问响应元数据（如 token 计数）。
    - 解析后的数据在"parsed"字段。解析的状态在"parsing_error"字段。

### (5) 嵌套结构化输出

In [6]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
class Actor(BaseModel):
    name: str = Field(description="演员姓名") 
    role: str = Field(description="演员扮演的角色或者扮演的角色姓名")
    
class Movie(BaseModel):
    """电影的详细描述"""
    title: str = Field("Gone With Wind", description="电影名")
    year: int = Field(default=2, description="电影发布的年份")
    director: str = Field("张导", description="电影拍摄的导演", title="导演")
    rating: float = Field(8.5, description="电影的评分")
    cast:list[Actor] = Field(description="演员表")

model = init_chat_model("ollama:qwen3:4b")
model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("关于中国大陆地区电影《1942》的信息")
print(response) 
print(type(response)) 

title='Gone With Wind' year=2012 director='张艺谋' rating=6.856257941176471 cast=[Actor(name='张艺谋', role='导演'), Actor(name='黄渤', role='演员'), Actor(name='李冰冰', role='演员')]
<class '__main__.Movie'>


In [8]:
from typing import List, Optional
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

# 1. 定义嵌套的 Pydantic 模型 (Schema)
class Address(BaseModel):
    """地址信息"""
    city: str = Field(description="所在城市")
    street: Optional[str] = Field(default=None, description="街道名称")

class Education(BaseModel):
    """教育经历"""
    degree: str = Field(description="获得的学位")
    major: str = Field(description="专业名称")
    graduation_year: int = Field(description="毕业年份")

class Resume(BaseModel):
    """简历信息"""
    name: str = Field(description="候选人的姓名")
    age: int = Field(description="候选人的年龄")
    address: Address = Field(description="候选人的居住地址")  # 嵌套模型
    education_list: List[Education] = Field(description="候选人的教育经历列表")  # 列表嵌套

# 2. 初始化支持结构化输出的模型
model = init_chat_model("ollama:qwen3.5:9b")
# 关键：用 with_structured_output 绑定 Schema
structured_model = model.with_structured_output(Resume)

# 4. 准备输入文本（非结构化数据）
text = """
我叫杨强，今年55岁。我住在南京市双龙大道60号陆军工程大学双龙营区。
教育背景：
1989年至1993年，就读于西南师范大学，读数学教育，获得学士学位。
1993年至1996年，就读于西南师范大学，读基础数学，获得硕士学位。
"""

# 5. 调用模型
try:
    result: Resume = structured_model.invoke(f"请从以下文本中提取简历信息：{text}")
    
    # 6. 访问结果（由于 result 是 Resume 对象，可以通过点号访问属性）
    print(f"姓名: {result.name}")
    print(f"年龄: {result.age}")
    print(f"地址: {result.address.city} {result.address.street or ''}")
    print("\n教育经历:")
    for edu in result.education_list:
        print(f"  - {edu.graduation_year}年: {edu.degree} (专业: {edu.major})")
        
except Exception as e:
    print(f"结构化输出失败: {e}")

姓名: 杨强
年龄: 55
地址: 南京市 双龙大道 60 号陆军工程大学双龙营区

教育经历:
  - 1993年: 学士学位 (专业: 数学教育)
  - 1996年: 硕士学位 (专业: 基础数学)


- 总结：
    - 上面结构体输出方法，明显使用BaseModel会更加灵活。
        - 验证：Pydantic 模型提供自动验证。TypedDict 和 JSON Schema 需要手动验证。

## 5.2. 智能体的结构化输出

- LangChain在create_agemt函数通过response_format 参数提供了结构化输出。
    - 在讲解智能体的时候，我们体验了结构体输出。
    - 而且create_agent还提供结构输出策略;
        - ToolStrategy:ToolStrategy 使用人工工具调用来生成结构化输出。这适用于任何支持工具调用的模型。当提供商原生结构化输出（通过 ProviderStrategy）不可用或不可靠时，应使用 ToolStrategy。
            - ToolStrategy 是 LangChain 中用于结构化输出的工具调用策略，它通过将你的Schema包装成一个“虚拟工具”，利用模型的工具调用能力来返回结构化参数。这种方式特别适用于不支持原生结构化输出的模型（如 DeepSeek、部分本地部署模型等）
        - ProviderStrategy：ProviderStrategy使用模型提供商原生的结构化输出生成功能。这种方法更可靠，但仅适用于支持原生结构化输出的提供商

### (1) ToolStrategy使用

In [10]:
# 1. 导入必要的库
from typing import List, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

# 2. 定义嵌套的 Pydantic 模型
class Address(BaseModel):
    """地址信息"""
    city: str = Field(description="所在城市")
    street: Optional[str] = Field(default=None, description="街道名称")

class Education(BaseModel):
    """教育经历"""
    degree: str = Field(description="获得的学位")
    major: str = Field(description="专业名称")
    graduation_year: int = Field(description="毕业年份")

class Resume(BaseModel):
    """简历信息 - 主模型（嵌套结构）"""
    name: str = Field(description="候选人的姓名")
    age: int = Field(description="候选人的年龄")
    address: Address = Field(description="候选人的居住地址")  # 嵌套模型
    education_list: List[Education] = Field(description="候选人的教育经历列表")  # 列表嵌套

# 3. 初始化模型（选择一个不支持原生结构化输出的模型来展示 ToolStrategy）
# 注意：使用支持工具调用的模型即可，ToolStrategy 会自动生效
model = init_chat_model("ollama:qwen3.5:9b")  # Qwen支持ToolStrategy

# 4. 显式使用 ToolStrategy 包装 Schema
# agent = create_agent(
#     model=model,
#     response_format=ToolStrategy(Resume),  # 关键：显式使用 ToolStrategy
#     system_prompt="你是一个专业的信息提取助手，请准确提取用户提供的简历信息。"
# )

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=Resume,
        tool_message_content="简历信息提取完成，数据已保存！",  # 自定义成功提示：自定义结构化输出生成后记录在对话历史中的提示信息
        handle_errors=True  # 自动处理错误并重试
    ),
    system_prompt="你是一个专业的信息提取助手，请准确提取用户提供的简历信息。"
)

# 5. 准备输入文本（非结构化数据）
text = """
我叫杨强，今年55岁。我住在南京市双龙大道60号陆军工程大学双龙营区。
教育背景：
1989年至1993年，就读于西南师范大学，读数学教育，获得学士学位。
1993年至1996年，就读于西南师范大学，读基础数学，获得硕士学位。
"""

# 6. 调用 Agent
result = agent.invoke({
    "messages": [{"role": "user", "content": f"请从以下文本中提取简历信息：{text}"}]
})

# 7. 获取结构化输出
resume: Resume = result["structured_response"]

# 8. 验证并访问结果
print(f"姓名: {resume.name}")
print(f"年龄: {resume.age}")
print(f"地址: {resume.address.city} {resume.address.street or ''}")
print("\n教育经历:")
for edu in resume.education_list:
    print(f"  - {edu.graduation_year}年: {edu.degree} (专业: {edu.major})")

# 也可以转为字典或 JSON 格式
print("\n原始 JSON 格式:")
print(resume.model_dump_json(indent=2))

姓名: 杨强
年龄: 55
地址: 南京市 双龙大道60号

教育经历:
  - 1993年: 学士学位 (专业: 数学教育)
  - 1996年: 硕士学位 (专业: 基础数学)

原始 JSON 格式:
{
  "name": "杨强",
  "age": 55,
  "address": {
    "city": "南京市",
    "street": "双龙大道60号"
  },
  "education_list": [
    {
      "degree": "学士学位",
      "major": "数学教育",
      "graduation_year": 1993
    },
    {
      "degree": "硕士学位",
      "major": "基础数学",
      "graduation_year": 1996
    }
  ]
}


In [11]:
result

{'messages': [HumanMessage(content='请从以下文本中提取简历信息：\n我叫杨强，今年55岁。我住在南京市双龙大道60号陆军工程大学双龙营区。\n教育背景：\n1989年至1993年，就读于西南师范大学，读数学教育，获得学士学位。\n1993年至1996年，就读于西南师范大学，读基础数学，获得硕士学位。\n', additional_kwargs={}, response_metadata={}, id='3a65263d-d9e9-45f5-9f9e-06af847ec7c0'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-11T15:40:41.3341091Z', 'done': True, 'done_reason': 'stop', 'total_duration': 35678777400, 'load_duration': 129402500, 'prompt_eval_count': 527, 'prompt_eval_duration': 485089400, 'eval_count': 335, 'eval_duration': 34840715900, 'logprobs': None, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--019d7d33-4695-7b93-aff0-43e11a28a3df-0', tool_calls=[{'name': 'Resume', 'args': {'name': '杨强', 'age': 55, 'address': {'campus': '陆军工程大学双龙营区', 'city': '南京市', 'street': '双龙大道60号'}, 'education_list': [{'degree': '学士学位', 'graduation_year': 1993, 'major': '数学教育'}, {'degree': '硕士学位', 'graduation_year': 1996, 'major':

- 代码说明：
    - ToolStrategy的工作原理：
        - 将Pydantic模型转换为一个虚拟工具（如extract-Resume），模型通过工具调用（tool call）返回结构化参数，LangChain 拦截该调用并验证数据。
        - 模型只需支持工具调用功能即可使用 ToolStrategy，大多数现代模型（DeepSeek、Claude、Llama 等）都支持
    - 与直接传入 Schema 的区别：
        - 当直接传入 response_format=Resume 时，LangChain 会根据模型能力自动选择策略：
            - 如果模型原生支持结构化输出（如 OpenAI GPT-4o）→ 使用 ProviderStrategy
            - 否则 → 自动降级为 ToolStrategy
        - 而显式使用 ToolStrategy 则强制使用工具调用策略，适用于：
            - 需要确保跨模型的一致性行为
            - 调试和测试场景
            - 模型虽然支持原生结构化输出，但你希望使用工具调用的特定特性（如自定义tool_message_content）

### (2) ProviderStrategy使用

- ProviderStrategy是LangChain中最可靠的结构化输出策略，它利用模型提供商的原生JSON Schema 支持（如 OpenAI 的 response_format、Anthropic 的 outputConfig 等）来强制模型输出符合预设格式的数据。这种方式比ToolStrategy更高效，且能提供更严格的schema校验:
    - ProviderStrategy 的工作原理:
        - 利用模型 API 原生的 response_format 或 outputConfig 机制，在 API 层面强制约束输出格式，模型返回的响应直接符合提供的 schema，无需额外的工具调用步骤.
        - ProviderStrategy 仅适用于原生支持结构化输出的模型：OpenAI GPT-4o/GPT-4o-mini/GPT-5、Anthropic Claude 3.5+、Google Gemini、xAI Grok 等
    - ProviderStrategy 的两种使用方式:
        - 显式使用 ProviderStrategy（推荐用于明确意图）
            - `agent = create_agent(model="...", response_format=ProviderStrategy(schema=Product, strict=True))  # 启用严格模式`
        - 隐式自动选择（由 LangChain 根据模型能力决定）
            - `agent = create_agent(model="...", response_format=Product)  # 自动选择 ProviderStrategy`


In [22]:
# 1. 导入必要的库
from typing import List, Optional
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.structured_output import ProviderStrategy
from pydantic import BaseModel, Field

# 2. 定义嵌套的 Pydantic 模型
class Manufacturer(BaseModel):
    """制造商信息"""
    name: str = Field(description="制造商名称")
    country: str = Field(description="制造商所在国家")
    website: Optional[str] = Field(default=None, description="制造商官网")

class Review(BaseModel):
    """产品评论"""
    username: str = Field(description="评论用户")
    rating: int = Field(description="评分，1-5分", ge=1, le=5)
    comment: str = Field(description="评论内容")

class Product(BaseModel):
    """产品信息 - 主模型"""
    name: str = Field(description="产品名称")
    price: float = Field(description="产品价格（美元）")
    manufacturer: Manufacturer = Field(description="制造商信息")  # 嵌套模型
    reviews: List[Review] = Field(default_factory=list, description="用户评论列表")  # 列表嵌套
    tags: List[str] = Field(description="产品标签")

# 3. 初始化支持原生结构化输出的模型
# ProviderStrategy 要求模型原生支持结构化输出，如 GPT-4o、GPT-4o-mini、Claude 3.5 Sonnet 等
model = init_chat_model("ollama:qwen3.5:9b", temperature=0, reasoning=False, enable_thinking=True)

# 关键：显式使用 ProviderStrategy 绑定 Schema
agent = create_agent(
    model=model,
    # response_format=ProviderStrategy(schema=Product),  # 显式使用 ProviderStrategy
    response_format=Product,
    system_prompt="你是一个专业的产品信息提取助手，请准确提取用户提供的产品信息。"
)

# 4. 准备输入文本（非结构化数据）
text = """
产品名称：MacBook Pro 14英寸
价格：1999美元

制造商信息：
- 公司名称：Apple Inc.
- 国家：美国
- 官网：www.apple.com

用户评论：
1. 用户"TechGuru"给了5星，评论："M3芯片性能强劲，续航出色，屏幕素质顶级！"
2. 用户"DailyUser"给了4星，评论："价格偏高，但整体体验很棒。"

标签：笔记本电脑、苹果、生产力工具
"""

# 5. 调用 Agent
result = agent.invoke({
    "messages": [{"role": "user", "content": f"请从以下文本中提取产品信息：{text}"}]
})

# 6. 获取结构化输出
product: Product = result["structured_response"]

# 7. 访问嵌套结构的结果
print(f"产品名称: {product.name}")
print(f"价格: ${product.price}")
print(f"\n制造商: {product.manufacturer.name}")
print(f"  国家: {product.manufacturer.country}")
print(f"  官网: {product.manufacturer.website or '无'}")

print("\n用户评论:")
for i, review in enumerate(product.reviews, 1):
    print(f"  {i}. {review.username}: {review.rating}星 - {review.comment}")

print(f"\n标签: {', '.join(product.tags)}")

# 也可以转为 JSON 格式查看完整输出
print("\n完整 JSON 格式:")
print(product.model_dump_json(indent=2))

产品名称: MacBook Pro 14英寸
价格: $1999.0

制造商: Apple Inc.
  国家: 美国
  官网: www.apple.com

用户评论:
  1. TechGuru: 5星 - M3芯片性能强劲，续航出色，屏幕素质顶级！
  2. DailyUser: 4星 - 价格偏高，但整体体验很棒。

标签: 笔记本电脑, 苹果, 生产力工具

完整 JSON 格式:
{
  "name": "MacBook Pro 14英寸",
  "price": 1999.0,
  "manufacturer": {
    "name": "Apple Inc.",
    "country": "美国",
    "website": "www.apple.com"
  },
  "reviews": [
    {
      "username": "TechGuru",
      "rating": 5,
      "comment": "M3芯片性能强劲，续航出色，屏幕素质顶级！"
    },
    {
      "username": "DailyUser",
      "rating": 4,
      "comment": "价格偏高，但整体体验很棒。"
    }
  ],
  "tags": [
    "笔记本电脑",
    "苹果",
    "生产力工具"
  ]
}


## 5.3. 智能体的消息解析处理

- 在langchain中模型与智能体的输出主要都采用BaseMessage。对模型的消息进行分析可以提起需要的信息。
    - 模型的输入格式：
        - PromptValue：提示值，一般来自提示词模版的生成。(支持管道)
        - str字符串
        - BaseMessages列表

- 模型的输出

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from langchain.tools import tool

model = init_chat_model(model="ollama:qwen3.5:9b")
question = "一个水池有一个进水管和一个出水管。进水管单独开需要4小时注满，出水管单独开需要6小时排空。两管同时开，几小时能注满？"
msg = {
    "role": "user", 
    "content": question
}
outputs = model.invoke(question)
print(type(outputs))
print(outputs)

<class 'langchain_core.messages.ai.AIMessage'>
content='这是一个经典的工程问题。我们可以通过计算“注水效率”和“排水效率”来解决。\n\n假设水池的总容量为 **1**。\n\n1.  **进水管的效率**：\n    每小时注水 $\\frac{1}{4}$。\n\n2.  **出水管的效率**：\n    每小时排水 $\\frac{1}{6}$。\n\n3.  **同时打开时的净注水效率**：\n    $$ \\frac{1}{4} - \\frac{1}{6} $$\n    通分后计算（公分母为 12）：\n    $$ \\frac{3}{12} - \\frac{2}{12} = \\frac{1}{12} $$\n\n    这意味着两管同时开时，每小时能净注满水池的 $\\frac{1}{12}$。\n\n4.  **所需时间**：\n    $$ 1 \\div \\frac{1}{12} = 12 \\text{（小时）} $$\n\n**答案：** 两管同时开，**12小时**能注满。' additional_kwargs={} response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-12T01:14:25.6441736Z', 'done': True, 'done_reason': 'stop', 'total_duration': 116644035700, 'load_duration': 127087700, 'prompt_eval_count': 50, 'prompt_eval_duration': 348282600, 'eval_count': 1070, 'eval_duration': 115241147300, 'logprobs': None, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'} id='lc_run--019d7f3f-5046-7213-9bb0-03db8a6586bf-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens

In [5]:
outputs = model.invoke([msg])
print(type(outputs))
print(outputs)

<class 'langchain_core.messages.ai.AIMessage'>
content='这是一个经典的工程问题。我们可以将水池的总容量看作单位“1”。\n\n**步骤如下：**\n\n1.  **确定进水速度：**\n    进水水管单独开需要4小时注满，所以进水速度是 $\\frac{1}{4}$ （池/小时）。\n\n2.  **确定出水速度：**\n    出水水管单独开需要6小时排空，所以出水速度是 $\\frac{1}{6}$ （池/小时）。\n\n3.  **确定同时打开时的净进水速度：**\n    当两管同时打开时，进水管在往里注水，出水管在往外排水，因此实际注水速度是进水速度减去出水速度：\n    $$ \\text{净速度} = \\frac{1}{4} - \\frac{1}{6} $$\n\n4.  **计算：**\n    通分计算：\n    $$ \\frac{1}{4} = \\frac{3}{12} $$\n    $$ \\frac{1}{6} = \\frac{2}{12} $$\n    $$ \\text{净速度} = \\frac{3}{12} - \\frac{2}{12} = \\frac{1}{12} $$\n    这意味着每过1小时，水池增加 $\\frac{1}{12}$ 的容量。\n\n5.  **计算所需时间：**\n    $$ \\text{时间} = \\frac{\\text{总工作量}}{\\text{净速度}} = \\frac{1}{\\frac{1}{12}} = 12 \\text{（小时）} $$\n\n**答案：**\n两管同时开，**12小时**能注满。' additional_kwargs={} response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-12T01:21:55.3001432Z', 'done': True, 'done_reason': 'stop', 'total_duration': 151425418400, 'load_duration': 135916600, 'prompt_eval_count': 50, 'prompt_eval_duration': 

In [11]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages([
    ("user", "{input}")
])

pval = template.invoke({"input": question})
print(type(pval))
print(pval)
outputs = model.invoke(pval)
print(type(outputs))
print(outputs)

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[HumanMessage(content='一个水池有一个进水管和一个出水管。进水管单独开需要4小时注满，出水管单独开需要6小时排空。两管同时开，几小时能注满？', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.messages.ai.AIMessage'>
content='这是一个经典的工程问题。我们可以通过计算“工作效率”来解决。\n\n设水池的总容量为 **1**。\n\n1.  **进水管的速度**：\n    单独需要4小时注满，所以每小时注入 $\\frac{1}{4}$。\n\n2.  **出水管的速度**：\n    单独需要6小时排空，所以每小时排出 $\\frac{1}{6}$。\n\n3.  **同时开启时的净速度**：\n    进水管在加水，出水管在排水，所以我们要用进水管的速度减去出水管的速度。\n    $$ \\text{净速度} = \\frac{1}{4} - \\frac{1}{6} $$\n\n4.  **通分计算**：\n    $$ \\frac{1}{4} = \\frac{3}{12} $$\n    $$ \\frac{1}{6} = \\frac{2}{12} $$\n    $$ \\frac{3}{12} - \\frac{2}{12} = \\frac{1}{12} $$\n    这意味着，两管同时开，每小时能净增加水池容量的 $\\frac{1}{12}$。\n\n5.  **计算所需时间**：\n    $$ \\text{时间} = \\frac{\\text{总工作量}}{\\text{净速度}} = 1 \\div \\frac{1}{12} = 12 $$\n\n**结论：**\n两管同时开，需要 **12小时** 才能注满水池。' additional_kwargs={} response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-12T01:31:12.5599515Z', 'done': True, '

- 代理的输出：

In [12]:
from langchain.agents import create_agent

agent = create_agent(model="ollama:qwen3.5:9b")
question = "一个水池有一个进水管和一个出水管。进水管单独开需要4小时注满，出水管单独开需要6小时排空。两管同时开，几小时能注满？"
msg = {
    "role": "user", 
    "content": question
}

response = agent.invoke({
    "messages": [msg]
})

print(type(response))
print(response)

<class 'dict'>
{'messages': [HumanMessage(content='一个水池有一个进水管和一个出水管。进水管单独开需要4小时注满，出水管单独开需要6小时排空。两管同时开，几小时能注满？', additional_kwargs={}, response_metadata={}, id='3e988809-a6e6-4b51-9492-b8560b28db5b'), AIMessage(content='这是一个经典的工程问题（水管问题）。我们可以通过计算每小时的注水效率来解决。\n\n**解题步骤：**\n\n1.  **设定工作效率**\n    假设水池的总容量为 $1$。\n    *   **进水管**单独开需要 4 小时注满，所以它的进水效率是：$\\frac{1}{4}$（每小时注满水池的 $\\frac{1}{4}$）。\n    *   **出水管**单独开需要 6 小时排空，所以它的出水效率是：$\\frac{1}{6}$（每小时排空水池的 $\\frac{1}{6}$）。\n\n2.  **计算同时打开时的实际效率**\n    当两管同时打开时，水池里的水一边在流入，一边在流出。\n    实际注水效率 = 进水效率 - 出水效率\n    $$ \\text{实际效率} = \\frac{1}{4} - \\frac{1}{6} $$\n\n3.  **通分计算**\n    为了相减，我们需要通分（公分母为 12）：\n    $$ \\frac{1}{4} = \\frac{3}{12} $$\n    $$ \\frac{1}{6} = \\frac{2}{12} $$\n    $$ \\text{实际效率} = \\frac{3}{12} - \\frac{2}{12} = \\frac{1}{12} $$\n    这意味着，两管同时开，每小时水池的水位上升相当于水池总容量的 $\\frac{1}{12}$。\n\n4.  **计算所需时间**\n    注满水池需要的时间 = 总容量 $\\div$ 实际注水效率\n    $$ \\text{时间} = 1 \\div \\frac{1}{12} = 12 \\text{（小时）} $$\n\n**结论：**\n当进水管和出水管同时打开时，需要 

In [16]:
response.keys()

dict_keys(['messages'])

In [25]:
response["messages"][-1].response_metadata

{'model': 'qwen3.5:9b',
 'created_at': '2026-04-12T01:40:48.3800886Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 522693150200,
 'load_duration': 130757300,
 'prompt_eval_count': 50,
 'prompt_eval_duration': 437168100,
 'eval_count': 4634,
 'eval_duration': 518592853200,
 'logprobs': None,
 'model_name': 'qwen3.5:9b',
 'model_provider': 'ollama'}

In [26]:
response["messages"][-1].usage_metadata

{'input_tokens': 50, 'output_tokens': 4634, 'total_tokens': 4684}

In [27]:
response["messages"][-1].tool_calls

[]

- 最后一个是智能体回答。

### (1) 推理内容提起

In [35]:
from typing import Optional, Dict, List, Any
from langchain_core.messages import BaseMessage
from langchain.messages import AIMessage
import re
def extract_reasoning(message: BaseMessage) -> Optional[Dict[str, Any]]:
    """
    提取推理过程
    
    某些模型（如 OpenAI o1）支持输出推理过程
    
    Args:
        message: LangChain 消息对象
        
    Returns:
        推理信息字典，包含 content 和 duration
    """
    if not isinstance(message, AIMessage):
        return None
    
    # 检查是否有推理相关的响应元数据
    response_metadata = getattr(message, "response_metadata", {})
    
    # OpenAI o1 模型的推理
    if "reasoning" in response_metadata:
        reasoning_data = response_metadata["reasoning"]
        return {
            "content": reasoning_data.get("content", ""),
            "duration": reasoning_data.get("duration_ms", 0) / 1000,  # 转换为秒
        }
    
    # 检查消息内容中是否包含 <thinking> 标签
    content = message.content
    if isinstance(content, str):
        thinking_match = re.search(r'<thinking>(.*?)</thinking>', content, re.DOTALL)
        if thinking_match:
            return {
                "content": thinking_match.group(1).strip(),
                "duration": 0,
            }
    
    return None

model = init_chat_model(
    model="ollama:qwen3.5:9b",
    reasoning=True
)
question = "一个水池有一个进水管和一个出水管。进水管单独开需要4小时注满，出水管单独开需要6小时排空。两管同时开，几小时能注满？"
msg = {
    "role": "user", 
    "content": question
}
outputs = model.invoke(question)
print(outputs)
extract_reasoning(outputs)

content='这是一个经典的“工程问题”（Work Problem）。我们可以通过计算水管的“工作效率”来解决。\n\n**解题步骤如下：**\n\n1.  **设水池的总容量为 "1"。**\n\n2.  **计算进水管的效率（注水速度）：**\n    *   进水管单独开需要 4 小时注满。\n    *   所以，每小时注水量为：$\\frac{1}{4}$。\n\n3.  **计算出水管的效率（排水速度）：**\n    *   出水管单独开需要 6 小时排空。\n    *   所以，每小时排水量为：$\\frac{1}{6}$。\n\n4.  **计算两管同时开启时的净注水速度：**\n    *   因为进水管在注水，出水管在排水，所以实际注水速度是两者之差。\n    *   净注水速度 = $\\frac{1}{4} - \\frac{1}{6}$。\n\n5.  **进行通分计算：**\n    *   $\\frac{1}{4} = \\frac{3}{12}$\n    *   $\\frac{1}{6} = \\frac{2}{12}$\n    *   净注水速度 = $\\frac{3}{12} - \\frac{2}{12} = \\frac{1}{12}$ （即每小时能注满水池的十二分之一）。\n\n6.  **计算注满所需的时间：**\n    *   时间 = 总工作量 ÷ 净注水速度\n    *   时间 = $1 \\div \\frac{1}{12} = 12$ 小时。\n\n**结论：**\n两管同时开，**12 小时**能注满水池。' additional_kwargs={'reasoning_content': 'Here\'s a thinking process that leads to the solution:\n\n1.  **Analyze the Problem:**\n    *   **Context:** A swimming pool (or tank) being filled and drained.\n    *   **Inlet Pipe (Inflow):** Can fill the pool alone in 4 hours.\n    *   **Outlet Pip

In [36]:
agent = create_agent(model=model)
response = agent.invoke({
    "messages": [msg]
})
print(response)
extract_reasoning(response["messages"][-1])

{'messages': [HumanMessage(content='一个水池有一个进水管和一个出水管。进水管单独开需要4小时注满，出水管单独开需要6小时排空。两管同时开，几小时能注满？', additional_kwargs={}, response_metadata={}, id='23a709ef-1ed2-485f-890c-db72444fa746'), AIMessage(content='这是一个经典的“工程问题”（Work Problem）。我们可以通过以下步骤来计算：\n\n1.  **设定总量**：\n    假设水池的总容量（工作总量）为 **1**。\n\n2.  **计算效率（速度）**：\n    *   **进水管的效率**：单独开4小时注满，说明每小时注水量为 $\\frac{1}{4}$。\n    *   **出水管的效率**：单独开6小时排空，说明每小时排水量为 $\\frac{1}{6}$。\n\n3.  **计算实际注水效率（净效率）**：\n    当两管同时打开时，水池的体积增加速度是“进水速度”减去“出水速度”。\n    $$ \\text{净效率} = \\text{进水效率} - \\text{出水效率} $$\n    $$ \\text{净效率} = \\frac{1}{4} - \\frac{1}{6} $$\n\n4.  **通分计算**：\n    为了方便计算，取4和6的最小公倍数12：\n    *   $\\frac{1}{4} = \\frac{3}{12}$\n    *   $\\frac{1}{6} = \\frac{2}{12}$\n    \n    所以：\n    $$ \\text{净效率} = \\frac{3}{12} - \\frac{2}{12} = \\frac{1}{12} $$\n\n    这意味着两管同时开时，每小时能净增加 $\\frac{1}{12}$ 池的水。\n\n5.  **计算所需时间**：\n    $$ \\text{时间} = \\text{工作总量} \\div \\text{净效率} $$\n    $$ \\text{时间} = 1 \\div \\frac{1}{12} = 12 \\text{（小时）} $$\n\n**结论：**\

In [37]:
print(response["messages"][-1].text)

这是一个经典的“工程问题”（Work Problem）。我们可以通过以下步骤来计算：

1.  **设定总量**：
    假设水池的总容量（工作总量）为 **1**。

2.  **计算效率（速度）**：
    *   **进水管的效率**：单独开4小时注满，说明每小时注水量为 $\frac{1}{4}$。
    *   **出水管的效率**：单独开6小时排空，说明每小时排水量为 $\frac{1}{6}$。

3.  **计算实际注水效率（净效率）**：
    当两管同时打开时，水池的体积增加速度是“进水速度”减去“出水速度”。
    $$ \text{净效率} = \text{进水效率} - \text{出水效率} $$
    $$ \text{净效率} = \frac{1}{4} - \frac{1}{6} $$

4.  **通分计算**：
    为了方便计算，取4和6的最小公倍数12：
    *   $\frac{1}{4} = \frac{3}{12}$
    *   $\frac{1}{6} = \frac{2}{12}$
    
    所以：
    $$ \text{净效率} = \frac{3}{12} - \frac{2}{12} = \frac{1}{12} $$

    这意味着两管同时开时，每小时能净增加 $\frac{1}{12}$ 池的水。

5.  **计算所需时间**：
    $$ \text{时间} = \text{工作总量} \div \text{净效率} $$
    $$ \text{时间} = 1 \div \frac{1}{12} = 12 \text{（小时）} $$

**结论：**
两管同时开，需要 **12小时** 能注满。


### (2) 提取工具调用信息

In [38]:
from typing import Optional, Dict, List, Any
from langchain_core.messages import BaseMessage
from langchain.messages import ToolMessage
import re
def extract_tool_calls(message: BaseMessage) -> List[Dict[str, Any]]:
    """
    提取工具调用信息
    
    Args:
        message: LangChain 消息对象
        
    Returns:
        工具调用列表
    """
    if not isinstance(message, AIMessage):
        return []
    
    # AIMessage 的 tool_calls 属性
    tool_calls = getattr(message, "tool_calls", [])
    
    if not tool_calls:
        return []
    
    result = []
    for tool_call in tool_calls:
        tool_info = {
            "id": tool_call.get("id", ""),
            "name": tool_call.get("name", ""),
            "type": f"tool-call-{tool_call.get('name', 'unknown')}",
            "state": "input-available",  # 初始状态
            "parameters": tool_call.get("args", {}),
            "result": None,
            "error": None,
        }
        result.append(tool_info)
    
    logger.debug(f"🔧 提取到 {len(result)} 个工具调用")
    return result

extract_tool_calls(response["messages"][-1])

[]

### (3) 抽取工具调用信息

In [40]:
def extract_tool_result(message: ToolMessage) -> Optional[Dict[str, Any]]:
    """
    提取工具执行结果
    
    Args:
        message: ToolMessage 对象
        
    Returns:
        工具结果信息
    """
    if not isinstance(message, ToolMessage):
        return None
    
    tool_call_id = getattr(message, "tool_call_id", "")
    content = message.content
    
    # 检查是否有错误
    is_error = getattr(message, "status", None) == "error"
    
    return {
        "id": tool_call_id,
        "state": "output-error" if is_error else "output-available",
        "result": None if is_error else content,
        "error": content if is_error else None,
    }
extract_tool_result(response["messages"][-1])

----